# Creating Spark Session

In [0]:
%pip install --upgrade pip
dbutils.library.restartPython()

%pip install beautifulsoup4 textacy nltk transformers SentencePiece tqdm torch spacy xformers fastcoref huggingface_hub
dbutils.library.restartPython()

%pip install --upgrade fastcoref
dbutils.library.restartPython()

import spacy
spacy.cli.download("en_core_web_sm")

dbutils.library.restartPython()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.config("spark.databricks.execution.timeout", "86400").config("spark.sql.session.timeZone", "UTC").config("spark.sql.execution.arrow.maxRecordsPerBatch", 32).appName("FinSentAnalysis").getOrCreate()

# Get Data

### Get News Data

In [0]:
# global constants
API_KEY : str = ''
BEZINGA_URL : str = 'https://api.benzinga.com/api/v2/news'
STOCK_TICKER : str = 'AAPL'
HEADERS = {"accept": "application/json"}
PAGE_SIZE = 100
PAGE_LIMIT = 400

params = {
    'token': API_KEY,
    'displayOutput' : 'full',
    'pageSize' : PAGE_SIZE,
    "sort": "created:asc",
    'tickers': STOCK_TICKER,
    'channels' : 'news'
}

In [0]:
import requests

news_data = []

params['dateFrom'] = '2015-01-01'
params['dateTo'] = '2020-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2021-01-01'
params['dateTo'] = '2023-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2024-01-01'
params['dateTo'] = '2025-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()


In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_news.json"

dbutils.fs.put(dbfs_path, json.dumps(news_data), overwrite=True)

### Get Stock Data

In [0]:
# global constants
API_KEY : str = ''
API_SECRET_KEY : str = ''
STOCK_TICKER : str = 'AAPL'
ALPACA_URL : str = f'https://data.alpaca.markets/v2/stocks/{STOCK_TICKER}/bars'
HEADERS = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET_KEY
    }

params = {
    'timeframe' : '1D',
    'limit' : 10000,
    'adjustment' : 'all'
}

In [0]:
import requests

stock_data = []

params['start'] = '2015-01-01'
params['end'] = '2018-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock1 = response.json()['bars']
stock_data += stock1

params['start'] = '2019-01-01'
params['end'] = '2021-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock2 = response.json()['bars']
stock_data += stock2

params['start'] = '2022-01-01'
params['end'] = '2025-11-05'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock3 = response.json()['bars']
stock_data += stock3

In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_price.json"

dbutils.fs.put(dbfs_path, json.dumps(stock_data), overwrite=True)

---

# Cataloging Data

In [0]:
import pandas as pd

workspace = 'danish.shahid@ucalgary.ca'

news_df = pd.read_json(f"../data/aapl_news.json")
price_df = pd.read_json(f"../data/aapl_price.json")

sp_news_df = spark.createDataFrame(news_df)
sp_price_df = spark.createDataFrame(price_df)

sp_news_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json", mode="overwrite")
sp_price_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json", mode="overwrite")

# Reading Data

In [0]:
news_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json").select(*['id', 'created', 'title', 'teaser', 'body'])
price_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json")

In [0]:
news_df.display()

# Curating Data

## Fixing Timestamps Types

In [0]:
news_df = news_df.withColumn('created', regexp_replace('created', r"^[A-Za-z]{3},\s+", "")).withColumn('created', to_timestamp('created', "dd MMM yyyy HH:mm:ss Z")).orderBy('created')
price_df = price_df.withColumn('t', to_timestamp('t', "yyyy-MM-ddTHH:mm:ssZ")).orderBy('t')

## Text Preprocessing

1. Remove HTML Tags
2. Remove New Line, Tab, Carriage Return
3. Replace URL, Emails, Phone Numbers, Emojis, Hashtags, Social User Handles
4. Normalize Bullet Points, Quotation Marks, Multi Line Hyphenation, and White Spaces
5. Remove 'Image' and 'Also Read:..'

In [0]:
# Remove HTML Tags
from bs4 import BeautifulSoup as bs

def parse_html(text: str) -> str:
  return bs(text, 'html.parser').get_text()

# Remove New Line, Tab, Carriage Return
import re
def remove_carriage(text: str) -> str:
  return re.sub(r'\r|\n|\t', ' ', text)

# Remove 'Image' and 'Also Read'
def replace_irrelevant(text: str) -> str:
    return re.sub(r'Image:.*|Also Read: ', '', text)

# Create a Textacy pipeline
from textacy.preprocessing import make_pipeline
from textacy.preprocessing.replace import emails, emojis, hashtags, phone_numbers, urls, user_handles
from textacy.preprocessing.normalize import bullet_points, quotation_marks, hyphenated_words, whitespace
text_pipe = make_pipeline(
    parse_html,
    remove_carriage,
    emails,
    emojis,
    hashtags,
    phone_numbers,
    urls,
    user_handles,
    bullet_points,
    quotation_marks,
    hyphenated_words,
    whitespace,
    replace_irrelevant
    )

# Convert into a Spark UDF
@udf
def text_preprocessing(text: str) -> str:
  return text_pipe(text)

In [0]:
news_df = news_df.withColumn('body', text_preprocessing('body')).withColumn('teaser', text_preprocessing('teaser')).withColumn('title', text_preprocessing('title'))

In [0]:
news_df.display()

## Coreference Resolution

### Local Model Cache

In [0]:
dbfs_local_tmp = "/Volumes/workspace/default/ensf612/hf_models/fcoref"

In [0]:
from huggingface_hub import snapshot_download
import os
import shutil

local_tmp = "/tmp/hf_models/fcoref"
os.makedirs(local_tmp, exist_ok=True)

# Download HF snapshot into a normal local folder
model_tmp = snapshot_download(
    "biu-nlp/f-coref",
    local_dir=local_tmp
)

shutil.copytree(model_tmp, dbfs_local_tmp, dirs_exist_ok=True)

### Pandas UDF Function

In [0]:
from fastcoref import FCoref
import pandas as pd

HF_COREF_CACHE_DIR = str(dbfs_local_tmp)

_coref = None

def get_coref():
    """
    Lazily initialize FCoref once per worker process.
    Called inside the pandas UDF.
    """
    global _coref
    if _coref is None:
        _coref = FCoref(
            model_name_or_path=HF_COREF_CACHE_DIR,
            device="cpu",              # serverless -> CPU
        )
    return _coref

def get_resolved_text(result) -> str:

    if result is None:
        return None

    """
    Build a "resolved" text by replacing later mentions in each cluster
    with the first mention's surface string.
    """
    text = result.text
    clusters = result.get_clusters(as_strings=False)  # [[(start, end), ...], ...]

    # Collect replacements: (start, end, replacement_text)
    replacements = []

    for cluster in clusters:
        if not cluster:
            continue

        # First span is the canonical mention
        canonical_start, canonical_end = cluster[0]
        canonical_text = text[canonical_start:canonical_end]

        # Replace all *later* mentions with canonical text
        for (start, end) in cluster[1:]:
            replacements.append((start, end, canonical_text))

    # Sort by start index so we can rebuild left→right
    replacements.sort(key=lambda x: x[0])

    # Rebuild the text with replacements applied
    resolved_parts = []
    cur = 0

    for start, end, rep in replacements:
        # add text before this mention
        resolved_parts.append(text[cur:start])
        # add canonical form
        resolved_parts.append(rep)
        # move cursor
        cur = end

    # add the tail of the text
    resolved_parts.append(text[cur:])

    return "".join(resolved_parts)


@pandas_udf(StringType())
def coreference_resolution(col: pd.Series) -> pd.Series:
    """
    Spark pandas UDF:
    - col: pandas.Series of strings from a Spark column
    - returns: pandas.Series of resolved strings
    """
    mask = col.isna()

    texts = col.fillna("").tolist()
    n = len(texts)
    resolved_all = []

    coref = get_coref()          # lazy init per worker
    batch_size = 4               # tune if needed

    for start in range(0, n, batch_size):
        batch = texts[start:start + batch_size]
        if not batch:
            continue

        # predict(list[str]) -> list[CorefResult]
        preds = coref.predict(texts=batch)
        resolved_batch = [get_resolved_text(pred) for pred in preds]
        resolved_all.extend(resolved_batch)

    out = pd.Series(resolved_all, index=col.index)
    # restore original nulls
    out[mask] = None
    return out


In [0]:
news_df = news_df.withColumn("body", coreference_resolution("body")).withColumn("title", coreference_resolution("title")).withColumn("teaser", coreference_resolution("teaser"))

news_df.count()

In [0]:
news_df.display()

## Contextual Sentence Segmentation

In [0]:
import en_core_web_sm

nlp = en_core_web_sm.load()

APPLE_NAMES = {
    "apple",
    "apple inc.",
    "apple, inc.",
    "apple incorporated",
}

def is_aapl_sentence(span):
    """
    Decide if a sentence is about Apple stock / company.
    Heuristics:
      - contains ticker 'AAPL'
      - or has ORG/PRODUCT entity with Apple name
    """
    text_lower = span.text.lower()

    # Check explicit ticker mention
    if "aapl" in text_lower or "apple" in text_lower:
        return True

    # Check NER entities
    for ent in span.ents:
        if ent.label_ in ("ORG", "PRODUCT"):
            if ent.text.lower() in APPLE_NAMES:
                return True

    return False

@udf
def split_sentences(text):
    doc = nlp(text)
    return '|'.join([sent.text.strip() for sent in doc.sents if is_aapl_sentence(sent)])

In [0]:
news_df = news_df.withColumn('body', split(split_sentences('body'), r'\|'))

news_df.count()

In [0]:
news_df.display()

## Silver Table Creation

In [0]:
news_df.write.format("delta").mode("overwrite").saveAsTable("aapl_news_curated")
#news_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news_curated.json", mode="overwrite")
#sp_price_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json", mode="overwrite")

## Read Silver Table

In [0]:
news_df = spark.read.table("aapl_news_curated")

# Sentiment Analysis

In [0]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

HF_CACHE_DIR = "dbfs:/Volumes/workspace/default/ensf612/hf_models/finbert"

_classifier = None

def get_classifier():
    """
    Lazily initialize the FinBERT pipeline on each worker.
    This function runs ON THE EXECUTOR, not the client,
    so the model is not serialized over gRPC.
    """
    global _classifier
    if _classifier is None:
        _classifier = pipeline("text-classification", model='ProsusAI/finbert', model_kwargs={'cache_dir' : HF_CACHE_DIR}, top_k=1, device=device)
    return _classifier

In [0]:
import pandas as pd
from collections import Counter
#from tqdm.auto import tqdm

@pandas_udf(ArrayType(MapType(StringType(), DoubleType())))
def classify_text_udf(bodies: pd.Series) -> pd.Series:
    """
    bodies: each row is a Python list of sentences (Spark array<string>)

    returns: each row is a list of maps:
      [
        {"positive": 0.1},
        {"positive": 0.6},
        ...
      ]
    """
    out = []
    clf = get_classifier()  # model created on worker the first time

    for sentences in bodies:
        # handle nulls
        if (
        sentences is None
        or (isinstance(sentences, str) and sentences.strip() == "")
        ):
            out.append(None)
            continue

        if isinstance(sentences, str):
            sentences = [sentences]
        
        try:
            # run FinBERT on the list of sentences
            preds = clf(
                sentences
            )
        except:
            print(f'Error : {sentences}')
        
        top_labels = []
        top_label_scores = []  # list of (label, score)
        for per_sentence in preds:
            # per_sentence is a list like:
            # [{"label": "positive", "score": ...}, {"label": "negative", ...}, ...]
            (label, score) = {item["label"]: float(item["score"]) for item in per_sentence}.items()
            top_labels.append(label)
            top_label_scores.append((label, score))

        # --- 3. combine per-sentence sentiments ---
        # most frequent top label
        label_counts = Counter(top_labels)
        combined_label = label_counts.most_common(1)[0][0]

        # mean score for that label across sentences where it was top
        scores_for_label = [score for lbl, score in top_label_scores if lbl == combined_label]
        combined_score = sum(scores_for_label) * 1.0 / len(scores_for_label)

        # final single-label map
        combined_sentiment = {combined_label: combined_score}
        out.append(combined_sentiment)

    return pd.Series(out)

In [0]:
news_df_part = news_df#.repartition(int((news_df.count()/20) + 1))
#news_df_part.count()
news_df_trunc = news_df_part.withColumn('sentiment_body', classify_text_udf('body')).withColumn('sentiment_title', classify_text_udf('title')).withColumn('sentiment_teaser', classify_text_udf('teaser'))

In [0]:
news_df_trunc.display()

In [0]:
news_df_trunc.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_sentiment")

# Resources

1. [https://arxiv.org/pdf/2306.02136](https://arxiv.org/pdf/2306.02136)